In [ ]:
import os
import time
import json
from dotenv import load_dotenv

load_dotenv()

from pydantic import BaseModel
import importlib

import agents

importlib.reload(agents)

from agents.templates.play_zero_agent import PlayZeroAgent
from agents.structs import FrameData, GameState

import textwrap

def print_wrapped_text(text: str, width: int = 80):
    """
    Print the given text with word-wrapped lines for better readability in the terminal.

    Args:
        text (str): The input text to be printed.
        width (int): The maximum line width before wrapping. Default is 80.
    """
    wrapper = textwrap.TextWrapper(width=width)
    paragraphs = text.strip().split("\n\n")

    for paragraph in paragraphs:
        wrapped = wrapper.fill(paragraph)
        print(wrapped + "\n")

#  Agent.__init__() missing 5 required positional arguments: 'card_id', 'game_id', 'agent_name', 'ROOT_URL', and 'record'
play_zero_agent: PlayZeroAgent = PlayZeroAgent(
    card_id="play_zero_agent",
    game_id="play_zero_agent",
    agent_name="PlayZeroAgent",
    ROOT_URL="http://localhost:8000",
    record=False,
)

runs = [
    {
        "game_id": "ls20",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track1.mp4",
        "scorecard_file_path": "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.8aefc1ea-8e65-41ec-9272-a8faff24ddb1.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\"",
    },
    {
        "game_id": "ls20",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track2.mp4",
        "scorecard_file_path": "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.fa1f97de-edb6-47c7-98bb-eff9f689d487.recording.jsonl",  
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\"",
    },
]

def get_frames(scorecard_file_path):
    with open(scorecard_file_path, "r") as file:
        grid_jsons = [json.loads(line) for line in file]
    frames = [FrameData(**frame_json["data"]) for frame_json in grid_jsons]
    return frames

def write_eval_log(evaluation_result):
    with open("eval.log", "a") as eval_log_file:
        print_wrapped_text(f"Evaluation Result:\n\n {evaluation_result}\n")
        eval_log_file.write(f"Evaluation Result:\n\n {evaluation_result}\n")

# Eval prompt for multiple_hypothesis_text

EVAL_PROMPT = """Give score and reason of whether the multiple hypothesis can be used to generate the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Multiple Hypothesis Text: <multiple_hypothesis_text>{multiple_hypothesis_text}</multiple_hypothesis_text>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""

GOAL_RELEVANCE_PROMPT = """Give score and reason of whether the generated goal can be used to achieve the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Generated Goal: <generated_goal>{generated_goal}</generated_goal>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""


def evaluate_multiple_hypothesis_text(multiple_hypothesis_text: str, expected_goal: str):
    prompt = EVAL_PROMPT.format(
        expected_goal=expected_goal,
        multiple_hypothesis_text=multiple_hypothesis_text
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data

def eval_goal_relevance(generated_goal: str, expected_goal: str):
    prompt = GOAL_RELEVANCE_PROMPT.format(
        expected_goal=expected_goal,
        generated_goal=generated_goal
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data

class RunResult(BaseModel):
    logical_analysis_actions_summary: str = ""
    eval_multiple_hypothesis_result: dict = {}
    eval_goal_relevance_result: dict = {}
    multiple_hypothesis_text: str = ""
    goal: str = ""

def store_run_results(run_results: list[dict]):
    # generate a unique path for the run results file
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_results_path = f"data/run_results_{timestamp}.json"
    with open(run_results_path, "w") as file:
        json.dump(run_results, file)


In [19]:

def test_run(run) -> RunResult:
    video_path = run["video_path"]
    scorecard_file_path = run["scorecard_file_path"]
    frame_start = run.get("frame_start", 0)
    frame_end = run.get("frame_end", None)
    expected_goal = run.get("expected_goal", "")

    frames = get_frames(scorecard_file_path)
    frames_for_analysis = frames[frame_start:frame_end] if frame_end else frames[frame_start:]

    print(f"Running analysis for video: {video_path}")
    print(f"Scorecard file: {scorecard_file_path}")
    print(f"Frames from {frame_start} to {frame_end if frame_end else 'end'}")

    logical_analysis_actions_summary = play_zero_agent.generate_logical_analysis_summary(frames_for_analysis)
    print(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")

    multiple_hypothesis_text = play_zero_agent.generate_multiple_random_hypothesis_from_video(
        video_file_path=video_path,
        logical_analysis_actions_summary=logical_analysis_actions_summary,
    )
    print_wrapped_text(f"Multiple Hypothesis Text:\n\n {multiple_hypothesis_text}")
    eval_multiple_hypothesis_result = evaluate_multiple_hypothesis_text(
        multiple_hypothesis_text=multiple_hypothesis_text,
        expected_goal=expected_goal,
    )
    write_eval_log(eval_multiple_hypothesis_result)
    goal = play_zero_agent.generate_top_hypothesis(
        multiple_hypothesis_text=multiple_hypothesis_text,
        logical_analysis_actions_summary=logical_analysis_actions_summary,
    )
    print_wrapped_text(f"Generated Goal:\n\n {goal}")
    eval_goal_relevance_result = eval_goal_relevance(
        generated_goal=goal,
        expected_goal=expected_goal
    )
    write_eval_log(eval_goal_relevance_result)


    return RunResult(
        logical_analysis_actions_summary=logical_analysis_actions_summary,
        eval_multiple_hypothesis_result=eval_multiple_hypothesis_result,
        eval_goal_relevance_result=eval_goal_relevance_result,
        multiple_hypothesis_text=multiple_hypothesis_text,
        goal=goal
    )

run_results = []
for run in runs[:1]:
    run_result = test_run(run)
    run_results.append({
        "run": run,
        "result": run_result.model_dump()
    })
store_run_results(run_results)

Running analysis for video: /workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track1.mp4
Scorecard file: /workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.8aefc1ea-8e65-41ec-9272-a8faff24ddb1.recording.jsonl
Frames from 0 to 55
Logical Analysis Actions Summary:

 Out of all user inputs recorded during gameplay:

- **W** was used 15 times, with 4 inputs having no effect on gameplay.
- **A** was used 7 times, and all had an effect on the game.
- **S** was used 5 times, with 2 inputs having no effect on gameplay.
- **D** was used 10 times, with 3 inputs having no effect on gameplay.
- **CLICK** was used 8 times, and all had no effect on the game.
Multiple Hypothesis Text:

 Here are the observed game effects, focusing on elements and the final goal:

The **final goal** observed is to move the **Orange_Rectangular_Block** (approx.
3x1 grid unit, orange rectangular block) to the top edge of the
**Larg